<a href="https://colab.research.google.com/github/sunonmountain/Revenue-Integrity-Engine/blob/main/03_AI_Revenue_Enrichment_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -U google-genai tavily-python pandas pydantic

import pandas as pd
import time
import json
import os
from google import genai
from google.genai import types
from tavily import TavilyClient
from pydantic import BaseModel, Field
from google.colab import userdata

# 1. Retrieve the key from Colab Secrets
api_key_gemini= userdata.get('GEMINI_API_KEY')
api_key_tavily= userdata.get('TAVILY_API_KEY')

# 2. Set the Gemini, Tavily API key as an environment variable
os.environ['GEMINI_API_KEY'] = api_key_gemini
os.environ['TAVILY_API_KEY'] = api_key_tavily


# FIX: Using the verified stable model identifier for the v1beta API
MODEL_ID = "gemini-3-flash-preview"

# --- 2. DATA MODELS & ROI TRACKING ---
class LeadIntelligence(BaseModel):
    job_title: str = Field(description="Current professional job title")
    intent_signal: str = Field(description="Growth trigger: hiring, funding, or expansion")
    confidence: float = Field(description="Accuracy score 0.0 to 1.0")

class RevenueROI:
    def __init__(self):
        self.credits_used = 0
        self.start_time = time.time()
        self.manual_cost_saved = 0.0

    def log_lead(self):
        self.credits_used += 2 # Advanced Tavily Search
        self.manual_cost_saved += 1.25 # Est. value of 5 mins of SDR research

    def report(self):
        return {
            "Total Search Credits": self.credits_used,
            "Total AI Requests": self.credits_used // 2,
            "Human Time Saved (Value)": f"${self.manual_cost_saved:.2f}",
            "Infrastructure Cost": "$0.00 (Free Tier)"
        }

# --- 3. THE ENRICHMENT ENGINE ---
class EnrichmentEngine:
    def __init__(self, tavily_api_key, gemini_api_key): # Modified to accept API keys
        self.tavily = TavilyClient(api_key=tavily_api_key)
        self.genai = genai.Client(api_key=gemini_api_key)
        self.roi = RevenueROI()
        print(f"--- Engine Ready | Model: {MODEL_ID} ---")

    def enrich(self, input_csv):
        if not os.path.exists(input_csv):
            print(f"❌ Error: {input_csv} not found.")
            return

        df = pd.read_csv(input_csv)
        # Target: High-value leads missing titles
        targets = df[df['Job_Title'].isna()].head(5) # Testing with first 5

        print(f"🚀 Pinging AI for {len(targets)} high-value leads...")

        for index, row in targets.iterrows():
            name = f"{row['First_Name']} {row['Last_Name']}"
            print(f"   f50d Researching: {name}...")

            try:
                # Phase 1: Search
                search = self.tavily.search(f"{name} {row['Company']} LinkedIn current role", search_depth="advanced")
                context = "\n".join([r['content'] for r in search['results']])

                # Phase 2: Intelligence
                response = self.genai.models.generate_content(
                    model=MODEL_ID,
                    contents=f"Lead: {name} @ {row['Company']}\nContext: {context}",
                    config=types.GenerateContentConfig(
                        system_instruction="Extract Job Title and Intent Signal as JSON.",
                        response_mime_type="application/json",
                        response_schema=LeadIntelligence
                    )
                )

                intel = json.loads(response.text)

                # Update DataFrame
                df.at[index, 'Job_Title'] = intel['job_title']
                df.at[index, 'Intent_Signal'] = intel['intent_signal']
                df.at[index, 'AI_Confidence'] = intel['confidence']

                self.roi.log_lead()
                print(f"   ✅ Success: {intel['job_title']}")
                time.sleep(1.2) # Rate limit protection

            except Exception as e:
                print(f"   ❌ Failed: {str(e)}")

        # Save to Gold Layer
        output_csv = 'Week3_Enriched_Data_Portfolio.csv'
        df.to_csv(output_csv, index=False)
        print("\n--- FINAL ROI REPORT ---")
        print(json.dumps(self.roi.report(), indent=4))

# --- 4. EXECUTION ---
engine = EnrichmentEngine(api_key_tavily, api_key_gemini) # Pass the retrieved API keys
engine.enrich('Week2_Cleaned_Data_Portfolio.csv')

--- Engine Ready | Model: gemini-3-flash-preview ---
🚀 Pinging AI for 5 high-value leads...
   f50d Researching: Gavin Vadivalloo...
   ✅ Success: SaaS and B2B Sales Professional
   f50d Researching: Mohammed Khan...
   ✅ Success: Chief Operating Officer
   f50d Researching: Emma Jones...
   ✅ Success: CEO and Founder
   f50d Researching: Alistair Higgins...
   ✅ Success: Digital Transformation Leader
   f50d Researching: Mohammed Khan...
   ✅ Success: CEO

--- FINAL ROI REPORT ---
{
    "Total Search Credits": 10,
    "Total AI Requests": 5,
    "Human Time Saved (Value)": "$6.25",
    "Infrastructure Cost": "$0.00 (Free Tier)"
}
